# Parkinson's Disease Voice Screening & Clinical Decision Support Platform
## Notebook 01: Data Acquisition, Quality Verification & Subject-Level Split

> **DISCLAIMER:** This software is a research screening tool, **NOT** a diagnostic device.

### Workflow Overview:
1. **Dependency Setup:** Install and import `datasets`, `huggingface_hub`, `soundfile`, `librosa`, `pandas`, `requests`, and `scikit-learn`.
2. **Dataset Acquisition:**
   - **Primary Dataset:** Italian Parkinson's Voice and Speech (IPVS) from Hugging Face mirror `birgermoell/Italian_Parkinsons_Voice_and_Speech` (no token needed, CC-BY 4.0, 831 recordings across 65 subjects).
   - **Secondary Dataset:** MDVR-KCL from Zenodo record `10.5281/zenodo.2867216` into `data/raw/mdvr_kcl/` (held out for cross-dataset evaluation).
3. **Metadata Construction:** Walk directory tree, map cohorts (`young_healthy`, `elderly_healthy`, `parkinsons`), parse filename prefixes into clinical speech tasks (`vowel_sustain`, `reading_passage`, `syllable_repetition`), and measure duration/sample rates.
4. **Data Quality Audit:** Verify 831 audio files open with `soundfile`, assert cohort counts, and flag anomalies.
5. **Subject-Level Split:** Group by `subject_id`, stratify by `label`, partition 70% Train / 15% Val / 15% Test with seed 42. Verify pairwise disjoint sets (`PASS`).
6. **Data Contract Generation:** Export `data/processed/metadata.csv` and `data/processed/split_manifest.json`.
7. **Exploratory Data Analysis (EDA):** Report class balance, per-task counts, duration statistics, and sample rate distributions.


In [1]:
# Cell 1: Install and import dependencies
import sys
import subprocess

required_packages = ["huggingface_hub", "soundfile", "librosa", "pandas", "requests", "tqdm", "scikit-learn"]
for pkg in required_packages:
    try:
        __import__(pkg.replace("-", "_"))
    except ImportError:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import os
import glob
import json
import zipfile
import shutil
from pathlib import Path
import requests
from tqdm import tqdm
import pandas as pd
import numpy as np
import soundfile as sf
from huggingface_hub import snapshot_download
from sklearn.model_selection import train_test_split

print("All dependencies successfully imported.")


All dependencies successfully imported.


In [2]:
# Cell 2: Download IPVS mirror (Hugging Face) and MDVR-KCL (Zenodo)
PROJECT_ROOT = Path(os.getcwd())
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent.parent

DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
IPVS_DIR = DATA_RAW_DIR / "ipvs"
MDVR_DIR = DATA_RAW_DIR / "mdvr_kcl"

DATA_RAW_DIR.mkdir(parents=True, exist_ok=True)
DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT.resolve()}")

# --- 1. Primary Dataset: IPVS (Hugging Face mirror) ---
print("\n[1/2] Downloading / Verifying Italian Parkinson's Voice and Speech (IPVS)...")
ipvs_path = snapshot_download(
    repo_id="birgermoell/Italian_Parkinsons_Voice_and_Speech",
    repo_type="dataset",
    local_dir=str(IPVS_DIR),
    local_dir_use_symlinks=False,
    resume_download=True
)
print(f"IPVS dataset located at: {ipvs_path}")

# --- 2. Secondary Dataset: MDVR-KCL (Zenodo Record 2867216) ---
# Held out strictly for cross-dataset evaluation; not used for training
print("\n[2/2] Downloading / Verifying MDVR-KCL held-out dataset from Zenodo...")
MDVR_DIR.mkdir(parents=True, exist_ok=True)
ZENODO_RECORD_URL = "https://zenodo.org/records/2867216/files/26_29_09_2017_KCL.zip?download=1"
mdvr_zip_path = MDVR_DIR / "26_29_09_2017_KCL.zip"

if not any(MDVR_DIR.glob("*.wav")):
    if not mdvr_zip_path.exists():
        print(f"Fetching MDVR-KCL archive from Zenodo ({ZENODO_RECORD_URL})...")
        response = requests.get(ZENODO_RECORD_URL, stream=True, timeout=60)
        response.raise_for_status()
        total_size = int(response.headers.get("content-length", 0))
        with open(mdvr_zip_path, "wb") as f, tqdm(
            desc="MDVR-KCL Download",
            total=total_size,
            unit="B",
            unit_scale=True,
            unit_divisor=1024,
        ) as bar:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
                    bar.update(len(chunk))
        print("Download complete.")

    print(f"Extracting {mdvr_zip_path.name}...")
    with zipfile.ZipFile(mdvr_zip_path, "r") as zip_ref:
        zip_ref.extractall(MDVR_DIR)
    print(f"MDVR-KCL extracted to {MDVR_DIR}")
else:
    print(f"MDVR-KCL recordings already present in {MDVR_DIR}")

n_ipvs_wav = len(list(IPVS_DIR.rglob("*.wav")))
n_mdvr_wav = len(list(MDVR_DIR.rglob("*.wav")))
print(f"\nAcquisition Summary:")
print(f"  - IPVS primary .wav files:     {n_ipvs_wav} (expected: 831)")
print(f"  - MDVR-KCL held-out .wav files: {n_mdvr_wav}")


Project root: /Users/eshwarsaielugam/Documents/prototype

[1/2] Downloading / Verifying Italian Parkinson's Voice and Speech (IPVS)...
IPVS dataset located at: /Users/eshwarsaielugam/Documents/prototype/data/raw/ipvs

[2/2] Downloading / Verifying MDVR-KCL held-out dataset from Zenodo...
MDVR-KCL recordings already present in /Users/eshwarsaielugam/Documents/prototype/data/raw/mdvr_kcl

Acquisition Summary:
  - IPVS primary .wav files:     831 (expected: 831)
  - MDVR-KCL held-out .wav files: 74


In [3]:
# Cell 3: Walk IPVS folder structure, inspect prefixes, and construct metadata table
#
# FILENAME PREFIX CONVENTION IN IPVS:
# 1. Vowel Sustains:
#    - VA1, VA2: sustained /a/ vowel (repetition 1 and 2)
#    - VE1, VE2: sustained /e/ vowel (repetition 1 and 2)
#    - VI1, VI2: sustained /i/ vowel (repetition 1 and 2)
#    - VO1, VO2: sustained /o/ vowel (repetition 1 and 2)
#    - VU1, VU2: sustained /u/ vowel (repetition 1 and 2)
# 2. Reading Passage:
#    - PR1 (or PR11): Prose reading passage ("Il brano")
# 3. Syllable Repetition & Diadochokinetic (DDK) tasks:
#    - B1, B2: Phonetic sentences and words (reading/syllable tasks)
#    - D1, D2: Diadochokinesia / rapid alternating syllable repetitions (e.g. /pa-ta-ka/)
#    - FB1: Frase breve / short phrase sequences

records = []
corrupt_files = []

# Find all audio files in the IPVS folder tree
ipvs_wav_files = sorted(list(IPVS_DIR.rglob("*.wav")))

for wav_path in tqdm(ipvs_wav_files, desc="Parsing Audio Metadata"):
    rel_path = wav_path.relative_to(PROJECT_ROOT)
    parts = wav_path.parts
    
    # Identify cohort folder index
    try:
        group_idx = next(i for i, p in enumerate(parts) if any(k in p for k in ["Young", "Elderly", "Parkinson"]))
    except StopIteration:
        continue
        
    group_str = parts[group_idx]
    fname = wav_path.name

    if "Young" in group_str:
        group = "young_healthy"
        label = 0
        sub_name = parts[group_idx + 1].strip().replace(" ", "_")
        subject_id = f"yhc_{sub_name}"
    elif "Elderly" in group_str:
        group = "elderly_healthy"
        label = 0
        sub_name = parts[group_idx + 1].strip().replace(" ", "_")
        subject_id = f"ehc_{sub_name}"
    elif "Parkinson" in group_str:
        group = "parkinsons"
        label = 1
        batch = parts[group_idx + 1].strip().replace(" ", "_")
        sub_name = parts[group_idx + 2].strip().replace(" ", "_")
        subject_id = f"pd_{batch}_{sub_name}"
    else:
        continue

    # Parse task type from prefix
    fname_upper = fname.upper()
    if fname_upper.startswith(("VA", "VE", "VI", "VO", "VU")):
        task_type = "vowel_sustain"
    elif fname_upper.startswith("PR"):
        task_type = "reading_passage"
    elif fname_upper.startswith(("B1", "B2", "D1", "D2", "FB1")):
        task_type = "syllable_repetition"
    else:
        task_type = "syllable_repetition"

    # Inspect audio integrity and header attributes
    try:
        info = sf.info(str(wav_path))
        sample_rate = info.samplerate
        duration_sec = round(info.duration, 4)
    except Exception as exc:
        corrupt_files.append((str(rel_path), str(exc)))
        continue

    records.append({
        "file_path": str(rel_path),
        "subject_id": subject_id,
        "group": group,
        "label": label,
        "task_type": task_type,
        "duration_sec": duration_sec,
        "sample_rate": sample_rate
    })

df_raw = pd.DataFrame(records)
print(f"Extracted metadata for {len(df_raw)} recordings across {df_raw['subject_id'].nunique()} subjects.")


Parsing Audio Metadata: 100%|██████████| 831/831 [00:01<00:00, 542.11it/s]
Extracted metadata for 831 recordings across 65 subjects.


In [4]:
# Cell 4: Quality assurance assertions and integrity verification
print("=== DATA QUALITY ASSURANCE AUDIT ===")

# 1. Check for corrupt files
if corrupt_files:
    print(f"WARNING: Found {len(corrupt_files)} corrupt files:")
    for path, err in corrupt_files:
        print(f"  - {path}: {err}")
else:
    print("✓ All 831 audio files opened successfully with soundfile (0 corrupt files).")

# 2. Assert file counts and cohort sizes
assert len(df_raw) == 831, f"Expected 831 files, got {len(df_raw)}"
print(f"✓ Total recording count assertion passed: {len(df_raw)} == 831")

assert df_raw["subject_id"].nunique() == 65, f"Expected 65 subjects, got {df_raw['subject_id'].nunique()}"
print(f"✓ Unique subject count assertion passed: {df_raw['subject_id'].nunique()} == 65")

cohort_counts = df_raw.groupby("group")["subject_id"].nunique().to_dict()
assert cohort_counts.get("young_healthy", 0) == 15, "Young healthy subjects != 15"
assert cohort_counts.get("elderly_healthy", 0) == 22, "Elderly healthy subjects != 22"
assert cohort_counts.get("parkinsons", 0) == 28, "Parkinsons subjects != 28"
print(f"✓ Cohort subject assertions passed: YHC={cohort_counts['young_healthy']}, EHC={cohort_counts['elderly_healthy']}, PD={cohort_counts['parkinsons']}")

# 3. Check for duration anomalies (zero duration or extreme outliers)
min_dur = df_raw["duration_sec"].min()
max_dur = df_raw["duration_sec"].max()
assert min_dur > 0, "Found audio file with <= 0 duration"
print(f"✓ Duration bounds valid: min={min_dur:.2f}s, max={max_dur:.2f}s")

# 4. Check sample rates
sample_rates = sorted(df_raw["sample_rate"].unique().tolist())
print(f"✓ Sample rates present: {sample_rates} Hz")
print("STATUS: ALL DATA QUALITY CHECKS PASSED.")


=== DATA QUALITY ASSURANCE AUDIT ===
✓ All 831 audio files opened successfully with soundfile (0 corrupt files).
✓ Total recording count assertion passed: 831 == 831
✓ Unique subject count assertion passed: 65 == 65
✓ Cohort subject assertions passed: YHC=15, EHC=22, PD=28
✓ Duration bounds valid: min=3.38s, max=250.31s
✓ Sample rates present: [16000, 44100] Hz
STATUS: ALL DATA QUALITY CHECKS PASSED.


In [5]:
# Cell 5: Subject-level stratified split (70% train, 15% val, 15% test, seed=42)
RANDOM_SEED = 42

# Extract subject-level label table (1 row per subject)
subjects_df = df_raw[["subject_id", "label"]].drop_duplicates().sort_values("subject_id").reset_index(drop=True)

# First split: 70% train vs 30% temp (val + test)
train_subj_df, temp_subj_df = train_test_split(
    subjects_df,
    test_size=0.30,
    random_state=RANDOM_SEED,
    stratify=subjects_df["label"]
)

# Second split: split temp 50/50 into val and test (15% each of total)
val_subj_df, test_subj_df = train_test_split(
    temp_subj_df,
    test_size=0.50,
    random_state=RANDOM_SEED,
    stratify=temp_subj_df["label"]
)

train_subs = sorted(train_subj_df["subject_id"].tolist())
val_subs = sorted(val_subj_df["subject_id"].tolist())
test_subs = sorted(test_subj_df["subject_id"].tolist())

set_train = set(train_subs)
set_val = set(val_subs)
set_test = set(test_subs)

# --- STRICT SUBJECT LEAKAGE AUDIT ---
intersect_train_val = set_train & set_val
intersect_train_test = set_train & set_test
intersect_val_test = set_val & set_test

print("=== SUBJECT LEAKAGE AUDIT ===")
print(f"Train subjects ({len(set_train)}) ∩ Val subjects ({len(set_val)}):   {len(intersect_train_val)} overlapping")
print(f"Train subjects ({len(set_train)}) ∩ Test subjects ({len(set_test)}):  {len(intersect_train_test)} overlapping")
print(f"Val subjects ({len(set_val)})   ∩ Test subjects ({len(set_test)}):  {len(intersect_val_test)} overlapping")

assert len(intersect_train_val) == 0, f"Subject leakage detected between Train and Val: {intersect_train_val}"
assert len(intersect_train_test) == 0, f"Subject leakage detected between Train and Test: {intersect_train_test}"
assert len(intersect_val_test) == 0, f"Subject leakage detected between Val and Test: {intersect_val_test}"

print("\nLEAKAGE CHECK: PASS (Zero subject overlap across all splits)")


=== SUBJECT LEAKAGE AUDIT ===
Train subjects (45) ∩ Val subjects (10):   0 overlapping
Train subjects (45) ∩ Test subjects (10):  0 overlapping
Val subjects (10)   ∩ Test subjects (10):  0 overlapping

LEAKAGE CHECK: PASS (Zero subject overlap across all splits)


In [6]:
# Cell 6: Map splits to recording rows and export data contracts
split_map = {s: "train" for s in train_subs}
split_map.update({s: "val" for s in val_subs})
split_map.update({s: "test" for s in test_subs})

df_raw["split"] = df_raw["subject_id"].map(split_map)

# Reorder columns to strictly match DATA CONTRACT:
# file_path, subject_id, group, label, task_type, duration_sec, sample_rate, split
CONTRACT_COLUMNS = [
    "file_path",
    "subject_id",
    "group",
    "label",
    "task_type",
    "duration_sec",
    "sample_rate",
    "split"
]
df_final = df_raw[CONTRACT_COLUMNS].sort_values(["split", "subject_id", "task_type"]).reset_index(drop=True)

# 1. Export metadata.csv
metadata_csv_path = DATA_PROCESSED_DIR / "metadata.csv"
df_final.to_csv(metadata_csv_path, index=False)
print(f"Saved: {metadata_csv_path} ({len(df_final)} rows)")

# 2. Build and export split_manifest.json
class_counts = {}
for s in ["train", "val", "test"]:
    sdf = df_final[df_final["split"] == s]
    class_counts[s] = {
        "recordings_total": int(len(sdf)),
        "recordings_healthy": int(sum(sdf["label"] == 0)),
        "recordings_parkinsons": int(sum(sdf["label"] == 1)),
        "subjects_total": int(sdf["subject_id"].nunique()),
        "subjects_healthy": int(sdf[sdf["label"] == 0]["subject_id"].nunique()),
        "subjects_parkinsons": int(sdf[sdf["label"] == 1]["subject_id"].nunique())
    }

split_manifest = {
    "train": train_subs,
    "val": val_subs,
    "test": test_subs,
    "seed": RANDOM_SEED,
    "class_counts": class_counts
}

manifest_path = DATA_PROCESSED_DIR / "split_manifest.json"
with open(manifest_path, "w") as f:
    json.dump(split_manifest, f, indent=2)
print(f"Saved: {manifest_path}")

# Verify file existence and column contract
assert metadata_csv_path.exists(), "metadata.csv was not written"
assert manifest_path.exists(), "split_manifest.json was not written"
reloaded_df = pd.read_csv(metadata_csv_path)
assert list(reloaded_df.columns) == CONTRACT_COLUMNS, f"Columns do not match contract: {list(reloaded_df.columns)}"
print("Data contracts verified successfully.")


Saved: /Users/eshwarsaielugam/Documents/prototype/data/processed/metadata.csv (831 rows)
Saved: /Users/eshwarsaielugam/Documents/prototype/data/processed/split_manifest.json
Data contracts verified successfully.


In [7]:
# Cell 7: Full Exploratory Data Analysis (EDA) Summary
print("=" * 70)
print("       PARKINSON'S VOICE PLATFORM: EXPLORATORY DATA ANALYSIS (EDA)")
print("=" * 70)

print("\n1. OVERALL DATASET METRICS")
print(f"  Total Audio Recordings: {len(df_final)}")
print(f"  Total Human Subjects:   {df_final['subject_id'].nunique()}")
print("  Cohort Breakdown:")
for grp, count in df_final.groupby("group").size().items():
    n_sub = df_final[df_final["group"] == grp]["subject_id"].nunique()
    print(f"    - {grp:18s}: {count:3d} recordings across {n_sub:2d} subjects")

print("\n2. CLASS BALANCE (SUBJECTS & RECORDINGS)")
n_healthy_sub = df_final[df_final["label"] == 0]["subject_id"].nunique()
n_pd_sub = df_final[df_final["label"] == 1]["subject_id"].nunique()
n_healthy_rec = sum(df_final["label"] == 0)
n_pd_rec = sum(df_final["label"] == 1)
print(f"  Subjects:   Healthy={n_healthy_sub} ({n_healthy_sub/65*100:.1f}%), Parkinsons={n_pd_sub} ({n_pd_sub/65*100:.1f}%)")
print(f"  Recordings: Healthy={n_healthy_rec} ({n_healthy_rec/len(df_final)*100:.1f}%), Parkinsons={n_pd_rec} ({n_pd_rec/len(df_final)*100:.1f}%)")

print("\n3. SPLIT DISTRIBUTION (SUBJECT-LEVEL STRATIFIED)")
split_rows = []
for s in ["train", "val", "test"]:
    sdf = df_final[df_final["split"] == s]
    split_rows.append({
        "Split": s.upper(),
        "Subjects Total": sdf["subject_id"].nunique(),
        "Subjects Healthy": sdf[sdf["label"] == 0]["subject_id"].nunique(),
        "Subjects PD": sdf[sdf["label"] == 1]["subject_id"].nunique(),
        "Recordings Total": len(sdf),
        "Recordings Healthy": sum(sdf["label"] == 0),
        "Recordings PD": sum(sdf["label"] == 1),
        "Healthy Ratio": f"{sum(sdf['label'] == 0)/len(sdf)*100:.1f}%"
    })
print(pd.DataFrame(split_rows).to_string(index=False))

print("\n4. TASK TYPE DISTRIBUTION")
for task, count in df_final.groupby("task_type").size().items():
    pct = count / len(df_final) * 100
    print(f"  - {task:22s}: {count:3d} recordings ({pct:4.1f}%)")

print("\n5. AUDIO DURATION STATISTICS (SECONDS)")
print(df_final["duration_sec"].describe().to_string())

print("\nDuration by Task Type (Mean +/- Std):")
for task, grp in df_final.groupby("task_type"):
    dur = grp["duration_sec"]
    print(f"  - {task:22s}: mean={dur.mean():.2f}s, std={dur.std():.2f}s, min={dur.min():.2f}s, max={dur.max():.2f}s")

print("\n6. SAMPLE RATE DISTRIBUTION")
for sr, count in df_final.groupby("sample_rate").size().items():
    print(f"  - {sr} Hz: {count:3d} recordings ({count/len(df_final)*100:.1f}%)")

print("=" * 70)


       PARKINSON'S VOICE PLATFORM: EXPLORATORY DATA ANALYSIS (EDA)

1. OVERALL DATASET METRICS
  Total Audio Recordings: 831
  Total Human Subjects:   65
  Cohort Breakdown:
    - elderly_healthy   : 349 recordings across 22 subjects
    - parkinsons        : 437 recordings across 28 subjects
    - young_healthy     :  45 recordings across 15 subjects

2. CLASS BALANCE (SUBJECTS & RECORDINGS)
  Subjects:   Healthy=37 (56.9%), Parkinsons=28 (43.1%)
  Recordings: Healthy=394 (47.4%), Parkinsons=437 (52.6%)

3. SPLIT DISTRIBUTION (SUBJECT-LEVEL STRATIFIED)
Split  Subjects Total  Subjects Healthy  Subjects PD  Recordings Total  Recordings Healthy  Recordings PD Healthy Ratio
TRAIN              45                26           19               584                 283            301         48.5%
  VAL              10                 5            5               139                  67             72         48.2%
 TEST              10                 6            4               108          